# UCI Hydraulic System Condition Monitoring
# UCI 液壓系統狀態監測

**Goal / 目標:** from raw sensor readings, tell whether the hydraulic rig's components are healthy or faulty, and rank cycles by risk.
從原始感測器讀數判斷液壓測試台的零件是否正常，並依風險排序。

**Data / 資料:** 2,205 cycles of 60 seconds, 17 sensors, 4 components (cooler, valve, pump, accumulator). Real sensors on a lab test rig, **faults were set deliberately** by the researchers. License: CC BY 4.0.
2,205 次、每次 60 秒的循環，17 個感測器，4 個零件（冷卻器、閥門、泵浦、蓄壓器）。真實感測器、實驗室測試台，**故障是研究人員刻意設定的**。授權 CC BY 4.0。

**Source / 來源:** UCI Machine Learning Repository, dataset 447 (Helwig, Pignanelli, Schütze). Please check the citation on the UCI page. / 引用格式請到 UCI 頁面核對。

> Chart labels are in English because Colab's default chart font cannot show Chinese.
> 圖表標籤使用英文，因為 Colab 預設的圖表字型無法顯示中文。

## 1. Download / 下載資料

The zip is about 73 MB. / 壓縮檔約 73 MB。

In [ ]:
!wget -q -O hydraulic.zip "https://archive.ics.uci.edu/static/public/447/condition+monitoring+of+hydraulic+systems.zip"
!unzip -q -o hydraulic.zip -d hydraulic
!ls -l hydraulic | head -30
!free -h | head -2

## 2. Features from raw readings / 從原始讀數整理特徵

Each sensor file has one row per cycle and one column per reading (6,000 for the 100 Hz sensors). We summarize each row into mean, standard deviation, min and max. Files are read one at a time to save memory.
每個感測器檔案是「一列一次循環、每欄一個讀數」（100 Hz 的感測器有 6,000 欄）。我們把每列整理成平均、標準差、最小、最大。檔案逐一讀取以節省記憶體。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sensors = ["PS1", "PS2", "PS3", "PS4", "PS5", "PS6", "EPS1", "FS1", "FS2",
           "TS1", "TS2", "TS3", "TS4", "VS1", "CE", "CP", "SE"]

feats = {}
for s in sensors:
    a = pd.read_csv(f"hydraulic/{s}.txt", sep="\t", header=None).values
    feats[f"{s}_mean"] = a.mean(axis=1)
    feats[f"{s}_std"] = a.std(axis=1)
    feats[f"{s}_min"] = a.min(axis=1)
    feats[f"{s}_max"] = a.max(axis=1)
    del a
X = pd.DataFrame(feats)
print("Feature table / 特徵表:", X.shape, " missing / 缺值:", int(X.isna().sum().sum()))
X.head()

## 3. Labels / 標籤

`profile.txt` has one row per cycle: cooler %, valve %, pump leakage, accumulator bar, stable flag.
`profile.txt` 每列對應一次循環：冷卻器 %、閥門 %、泵浦內漏、蓄壓器 bar、穩定旗標。

**"Faulty" is our own definition:** at least one component is not in its best state (cooler 100, valve 100, pump 0, accumulator 130). Change it if your project needs a different one.
**「有問題」是我們自己的定義：** 至少一個零件不在最佳狀態。若專案需要不同定義，請修改。

The `stable` flag is **not** used as a feature because we have not confirmed what it means (read `documentation.txt`).
`stable` 旗標**不當特徵**，因為我們還沒確認它的意思（請讀 `documentation.txt`）。

In [ ]:
prof = pd.read_csv("hydraulic/profile.txt", sep="\t", header=None,
                   names=["cooler", "valve", "pump", "accumulator", "stable"])

healthy = (prof.cooler == 100) & (prof.valve == 100) & (prof.pump == 0) & (prof.accumulator == 130)
y = (~healthy).astype(int)

print("Healthy cycles / 全部正常的循環:", int(healthy.sum()))
print("Faulty cycles / 有問題的循環:", int(y.sum()), " share / 比例:", round(y.mean(), 3))
for c in ["cooler", "valve", "pump", "accumulator"]:
    print(f"\n{c}:"); print(prof[c].value_counts().sort_index())

# How long are runs of identical labels? Long runs mean neighbouring cycles are near-copies.
# 相同標籤連續出現多長？連續很長代表相鄰的循環幾乎一樣。
combo = prof[["cooler", "valve", "pump", "accumulator"]].astype(str).agg("-".join, axis=1)
run_id = (combo != combo.shift()).cumsum()
runs = combo.groupby(run_id).size()
print("\nNumber of runs / 連續段數:", len(runs), " median length / 中位長度:", int(runs.median()), " max / 最長:", int(runs.max()))

## 4. Fair evaluation: random split vs. blocked split / 公平評估：隨機切分 vs. 區塊切分

Neighbouring cycles are very similar. A random split puts near-copies into both train and test, which can make scores look too good. A **blocked** split keeps consecutive cycles together. Compare both.
相鄰的循環非常相似。隨機切分會把幾乎一樣的資料同時放進訓練與測試，分數可能虛高。**區塊**切分讓連續的循環留在同一邊。兩種都比較。

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score, cross_val_predict

groups = np.arange(len(X)) // 100          # blocks of 100 consecutive cycles / 每 100 次連續循環為一區塊
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                            min_samples_leaf=2, random_state=42, n_jobs=-1)

rand_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
block_cv = GroupKFold(n_splits=5)

s_rand = cross_val_score(rf, X, y, cv=rand_cv, scoring="average_precision", error_score=np.nan)
s_block = cross_val_score(rf, X, y, cv=block_cv, groups=groups, scoring="average_precision", error_score=np.nan)

print("PR-AUC random split / 隨機切分:", s_rand.round(3), " mean / 平均:", round(np.nanmean(s_rand), 3))
print("PR-AUC blocked split / 區塊切分:", s_block.round(3), " mean / 平均:", round(np.nanmean(s_block), 3))
print("Baseline (share of faulty) / 基準（有問題的比例）:", round(y.mean(), 3))

## 5. Risk ranking: cumulative capture curve / 風險排序：累積捕捉曲線

Out-of-fold probabilities from the blocked split. If we inspect only the top k% of cycles by risk, what share of faulty cycles do we catch?
使用區塊切分的 out-of-fold 機率。只檢查風險最高的前 k% 循環，能抓到多少比例的有問題循環？

In [ ]:
p = cross_val_predict(rf, X, y, cv=block_cv, groups=groups, method="predict_proba")[:, 1]

order = np.argsort(-p)
caught = np.cumsum(np.asarray(y)[order]) / y.sum()
share = np.arange(1, len(p) + 1) / len(p)

plt.figure(figsize=(7, 5))
plt.plot(share, caught, label="RandomForest (blocked CV)")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Random inspection")
plt.xlabel("Share of cycles inspected"); plt.ylabel("Share of faulty cycles caught")
plt.title("Cumulative capture curve (hydraulic)"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 6. Which sensors matter? / 哪些感測器最重要？

In [ ]:
rf.fit(X, y)
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.tail(15).plot.barh(figsize=(7, 6))
plt.xlabel("Feature importance"); plt.title("Top 15 features (faulty vs. healthy)"); plt.tight_layout(); plt.show()

## 7. Each component separately / 各零件分開預測

Multi-class accuracy and macro-F1 with the blocked split. Macro-F1 treats every state equally.
用區塊切分算多分類的 accuracy 與 macro-F1。macro-F1 對每個狀態一視同仁。

In [ ]:
rows = []
for t in ["cooler", "valve", "pump", "accumulator"]:
    clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    acc = cross_val_score(clf, X, prof[t], cv=block_cv, groups=groups, scoring="accuracy", error_score=np.nan)
    f1 = cross_val_score(clf, X, prof[t], cv=block_cv, groups=groups, scoring="f1_macro", error_score=np.nan)
    rows.append([t, np.nanmean(acc), np.nanmean(f1), prof[t].value_counts(normalize=True).max()])
pd.DataFrame(rows, columns=["component", "accuracy", "macro_F1", "majority_class_share"]).round(3)

## 8. Limits to state in the report / 報告中要寫明的限制

- Real sensors, but a **lab test rig with deliberately set faults**, not natural failures in a factory. / 真實感測器，但是**刻意設定故障的實驗室測試台**，不是工廠中自然發生的故障。
- The task is **condition assessment** (what state is it in now), not predicting when something will fail. / 任務是**狀態判斷**（現在是什麼狀態），不是預測何時故障。
- "Faulty" is our own definition. / 「有問題」是我們自己的定義。
- Neighbouring cycles are near-copies, so compare random and blocked results and report the blocked one. / 相鄰循環幾乎相同，所以要比較兩種切分，並以區塊切分的結果為準。

## To do / 待辦
- [ ] Read `documentation.txt` and confirm what the `stable` flag means / 讀 documentation.txt，確認 stable 的意思
- [ ] Compare with the AI4I results in one table / 與 AI4I 結果放在同一張表比較
- [ ] Try fewer features (the literature reports a small set is enough) / 試用較少的特徵
- [ ] Add SHAP like in the AI4I notebook / 加入 SHAP